In [2]:
!pip install trl peft bitsandbytes accelerate transformers datasets

from huggingface_hub import login
login()   # entre ton token ici


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 465.5/465.5 kB 7.7 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 12.0 MB/s eta 0:00:00:00:0100:01


In [7]:
# ---------------- Notebook-ready SFT (complete) ----------------
# Installez d'abord (une seule fois) si nécessaire :
# !pip install -q trl==0.25.1 peft bitsandbytes accelerate transformers datasets sentencepiece huggingface_hub

import os
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    DataCollatorForLanguageModeling,
)
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTTrainer
from huggingface_hub import login

# ---------------- Hugging Face login (interactive)
print("Login to Hugging Face (paste token when prompted)...")
login()  # colle ton token HuggingFace (scope read)

# ---------------- Config générale (modèle)
# Choisis un modèle adapté à ton environnement.
# Si tu es sur un GPU Colab T4/L4/A100, TinyLlama fonctionne ; sinon utilises Qwen2-0.5B pour plus de robustesse.
BASE_MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
# BASE_MODEL_ID = "Qwen/Qwen2-0.5B-Instruct"  # alternative plus légère

CHAT_TEMPLATE = (
    "<s>[INST] <<SYS>>\n"
    "{system_prompt}\n"
    "<</SYS>>\n\n"
    "{instruction} [/INST] {response}</s>"
)

def formatting_prompts_func(example):
    """Convertit un batch JSONL → texte formatté"""
    texts = []
    # Gère les cas list/scalar
    if isinstance(example.get("instruction"), list):
        length = len(example["instruction"])
        for i in range(length):
            formatted = CHAT_TEMPLATE.format(
                system_prompt=example["system_prompt"][i],
                instruction=example["instruction"][i],
                response=example["response"][i],
            )
            texts.append(formatted)
    else:
        formatted = CHAT_TEMPLATE.format(
            system_prompt=example.get("system_prompt", ""),
            instruction=example.get("instruction", ""),
            response=example.get("response", ""),
        )
        texts.append(formatted)
    return {"text": texts}

def make_safe_data_collator(tokenizer):
    """Retourne un collator qui supprime 'text' avant le padding/tokenization."""
    base_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

    def collator(features):
        # sécurité : supprimer 'text' si présent dans les features
        if isinstance(features, (list, tuple)) and len(features) > 0 and isinstance(features[0], dict):
            for f in features:
                f.pop("text", None)
        return base_collator(features)

    return collator

def run_sft_training(dataset_path, output_dir, agent_name):
    print("\n" + "="*60)
    print(f"START SFT → agent: {agent_name}")
    print(f"Dataset: {dataset_path}")
    print("="*60 + "\n")

    # Vérifier présence du fichier (chemin RELATIF au CWD du notebook/script)
    if not os.path.exists(dataset_path):
        raise FileNotFoundError(f"[ERREUR] Dataset introuvable : {os.path.abspath(dataset_path)}")

    # Detect GPU availability
    has_cuda = torch.cuda.is_available()
    print("GPU disponible :", has_cuda)
    if has_cuda:
        torch_dtype = torch.float16
        device_map = "auto"
    else:
        torch_dtype = torch.float32
        device_map = {"": "cpu"}

    # ---------------- Model load (safe)
    try:
        # Charger le modèle SANS quantization (prévenir ImportError avec trust_remote_code)
        model = AutoModelForCausalLM.from_pretrained(
            BASE_MODEL_ID,
            device_map=device_map,
            trust_remote_code=True,
            torch_dtype=torch_dtype
        )
    except Exception as e_load:
        # fallback: essayer un modèle alternatif plus petit
        print("Warning: échec du chargement du modèle principal :", e_load)
        alt = "Qwen/Qwen2-0.5B-Instruct"
        print(f"Tentative de fallback vers {alt} ...")
        model = AutoModelForCausalLM.from_pretrained(
            alt,
            device_map=device_map,
            trust_remote_code=True,
            torch_dtype=torch_dtype
        )

    # Préparer le modèle pour k-bit training (compatible QLoRA style)
    # (utile même si tu n'appliques pas immédiatement bitsandbytes quantization)
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
    model.config.use_cache = False

    # ---------------- Tokenizer
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"
    model.tokenizer = tokenizer  # TRL 0.25.1 expects tokenizer attached to model

    # ---------------- Dataset
    # Utiliser data_files mapping pour éviter confusions de chemins
    dataset = load_dataset("json", data_files={"train": dataset_path})["train"]
    dataset = dataset.map(formatting_prompts_func, batched=True)
    # garder uniquement 'text'
    dataset = dataset.remove_columns([c for c in dataset.column_names if c != "text"])

    print(f"Nombre d'exemples : {len(dataset)}")
    print("Exemple (premier) :", dataset[0])

    # ---------------- PEFT (LoRA) config
    peft_config = LoraConfig(
        r=64,
        lora_alpha=16,
        lora_dropout=0.1,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    )

    # ---------------- Training args (ajuste si OOM)
    args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=3,
        per_device_train_batch_size=2,    # réduire si OOM
        gradient_accumulation_steps=2,
        learning_rate=2e-4,
        optim="paged_adamw_32bit",
        save_steps=500,
        logging_steps=50,
        fp16=has_cuda,                    # fp16 seulement si GPU
        report_to="none",
        remove_unused_columns=False,      # on nettoie via collator
    )

    safe_collator = make_safe_data_collator(tokenizer)

    trainer = SFTTrainer(
        model=model,
        args=args,
        train_dataset=dataset,
        peft_config=peft_config,
        formatting_func=lambda batch: batch["text"],  # TRL 0.25.1
        data_collator=safe_collator,
    )

    # ---------------- Train
    trainer.train()

    # ---------------- Save
    print(f"\nSaving model → {output_dir}")
    trainer.model.save_pretrained(output_dir)
    tokenizer.save_pretrained(output_dir)
    print(f"=== SFT FINI pour {agent_name} ===\n")



Login to Hugging Face (paste token when prompted)...


In [8]:
# ---------------- Run for all agents (paths RELATIFS au CWD)
agents = ["orchestrator", "researcher", "code_writer", "critic"]
for agent in agents:
    ds = f"data/processed_sft/{agent}_sft.jsonl"
    out = f"checkpoints/{agent}_lora"
    os.makedirs(out, exist_ok=True)
    try:
        run_sft_training(ds, out, agent)
    except Exception as e:
        print(f"\n--- Erreur durant l'entraînement de {agent} ---")
        print(e)
        continue

# ---------------- End
# ---------------- Fin du script ----------------


START SFT → agent: orchestrator
Dataset: data/processed_sft/orchestrator_sft.jsonl


--- Erreur durant l'entraînement de orchestrator ---
[ERREUR] Dataset introuvable : /content/data/processed_sft/orchestrator_sft.jsonl

START SFT → agent: researcher
Dataset: data/processed_sft/researcher_sft.jsonl


--- Erreur durant l'entraînement de researcher ---
[ERREUR] Dataset introuvable : /content/data/processed_sft/researcher_sft.jsonl

START SFT → agent: code_writer
Dataset: data/processed_sft/code_writer_sft.jsonl


--- Erreur durant l'entraînement de code_writer ---
[ERREUR] Dataset introuvable : /content/data/processed_sft/code_writer_sft.jsonl

START SFT → agent: critic
Dataset: data/processed_sft/critic_sft.jsonl


--- Erreur durant l'entraînement de critic ---
[ERREUR] Dataset introuvable : /content/data/processed_sft/critic_sft.jsonl


In [9]:
import os
print("Working directory:", os.getcwd())
print("Files:", os.listdir())


Working directory: /content
Files: ['.config', 'checkpoints', 'sample_data']


In [10]:
import glob
print(glob.glob("**/*.jsonl", recursive=True))


[]
